# 05 — Hyperparameter Tuning

## Purpose

Tune the XGBoost model (the flagship model for this project) using three
stages, as required by the project guideline:

1. **Random Search** — sample 60 parameter combinations across wide ranges.
2. **Grid Search** — exhaustively test a focused grid around the Random Search best.
3. **Manual tuning** — hand-picked extreme configurations to probe the overfitting boundary.

## Methodology

- **Cross-validation**: `TimeSeriesSplit` with 5 folds on the training set only.
  Each fold's validation comes strictly *after* its training — no future data leaks in.
- **Note on panel data**: `TimeSeriesSplit` splits by *row order*, not by year.
  To ensure chronological folds, we sort the panel by year (not by country)
  *before* passing it to CV. This makes each fold's validation set a block of
  later years — an unusual but correct way to apply `TimeSeriesSplit` to a
  country-year panel.
- **Scoring metric**: negative MAE. We report MAE in original units (percentage points).
- **Test set untouched**: The 2020–2024 test period is used only at the very end
  to compare the tuned models against the baseline.

## Objective

The purpose of tuning is **not only** to find better parameters, but to understand
how tuning affects model performance and overfitting. A model that improves
training fit but worsens test fit has not been improved in a useful sense.

In [13]:
import sys
from pathlib import Path

current = Path.cwd()
project_root = current.parent if current.name == "notebooks" else current
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [14]:
import pandas as pd
import numpy as np
from scipy.stats import uniform, randint
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src import config, split, models

from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

## 1. Data preparation

We load the rebuilt model dataset and sort by year (not by country).
This ensures `TimeSeriesSplit` produces chronologically clean folds:
earlier years → training, later years → validation.

The chronological train/test split from `preprocessing` is preserved:
train = 1995–2019, test = 2020–2024.

In [15]:
df = pd.read_csv(config.MODEL_DATA_PATH)
df = df.sort_values([config.YEAR_COL, config.COUNTRY_CODE_COL]).reset_index(drop=True)

X_train, y_train, X_test, y_test = split.get_train_test(df)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Train year range:", df[df[config.SPLIT_COL]=="train"][config.YEAR_COL].min(),
      "-", df[df[config.SPLIT_COL]=="train"][config.YEAR_COL].max())

Train: (675, 11)  Test: (135, 11)
Train year range: 1995 - 2019


## 2. Search space

We tune six XGBoost hyperparameters, chosen for their impact on the
bias-variance tradeoff:

| Parameter | Role | Range |
|---|---|---|
| `n_estimators` | Number of boosting rounds | 100–800 |
| `learning_rate` | Step size per round | 0.01–0.30 |
| `max_depth` | Tree depth (model complexity) | 2–8 |
| `subsample` | Row sampling per tree | 0.6–1.0 |
| `colsample_bytree` | Feature sampling per tree | 0.6–1.0 |
| `reg_lambda` | L2 regularization strength | 0.0–5.0 |

**Intuition**: `max_depth` and `n_estimators` control *capacity*;
`learning_rate` controls *speed*; `subsample`, `colsample_bytree`
and `reg_lambda` control *regularization*.

In [16]:
# Fresh XGBoost pipeline for tuning
def make_xgb_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
        ("model",   XGBRegressor(
            random_state=config.RANDOM_STATE,
            n_jobs=-1,
            verbosity=0,
        )),
    ])

# Search space for Random Search
param_dist = {
    "model__n_estimators":     randint(100, 800),
    "model__learning_rate":    uniform(0.01, 0.29),
    "model__max_depth":        randint(2, 8),
    "model__subsample":        uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.6, 0.4),
    "model__reg_lambda":       uniform(0.0, 5.0),
}

cv = split.get_time_series_cv(n_splits=5)
print("Param space ready. CV folds:", cv.get_n_splits())

Param space ready. CV folds: 5


## 3. Random Search

`RandomizedSearchCV` samples 60 parameter combinations and evaluates each
with 5-fold `TimeSeriesSplit` cross-validation. This is faster than Grid
Search over the same space and often finds equally good regions.

**Metric**: negative MAE (scikit-learn convention — higher is better; we
negate when reporting).

In [17]:
random_search = RandomizedSearchCV(
    estimator=make_xgb_pipeline(),
    param_distributions=param_dist,
    n_iter=60,
    scoring="neg_mean_absolute_error",
    cv=cv,
    random_state=config.RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

random_search.fit(X_train, y_train)

print("\nBest params (Random Search):")
for k, v in random_search.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nBest CV MAE: {-random_search.best_score_:.4f}")

Fitting 5 folds for each of 60 candidates, totalling 300 fits

Best params (Random Search):
  model__colsample_bytree: 0.6111981461958247
  model__learning_rate: 0.0691581295024103
  model__max_depth: 7
  model__n_estimators: 121
  model__reg_lambda: 3.697847710211919
  model__subsample: 0.7634181389255035

Best CV MAE: 2.4916


### Random Search — result

The best configuration found by Random Search:

| Parameter | Value |
|---|---|
| `colsample_bytree` | 0.61 |
| `learning_rate` | 0.069 |
| `max_depth` | 7 |
| `n_estimators` | 121 |
| `reg_lambda` | 3.70 |
| `subsample` | 0.76 |

**Best cross-validated MAE: 2.4916** (percentage points).

This is a substantial improvement over the untuned baseline, whose
cross-validated performance we can compare against in the table below.

Note that CV MAE is measured on the training years (via folds), not on
the held-out test period — final test evaluation comes at the end of
this notebook.

In [18]:
def report(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train) if not hasattr(model, "best_estimator_") else None
    est = model.best_estimator_ if hasattr(model, "best_estimator_") else model
    train_pred = est.predict(X_train)
    test_pred  = est.predict(X_test)
    return {
        "model":      name,
        "train_MAE":  mean_absolute_error(y_train, train_pred),
        "test_MAE":   mean_absolute_error(y_test,  test_pred),
        "train_R2":   r2_score(y_train, train_pred),
        "test_R2":    r2_score(y_test,  test_pred),
    }

rows = []
rows.append(report("XGB baseline (no tuning)", models.build_xgboost_baseline(), X_train, y_train, X_test, y_test))
rows.append(report("XGB RandomSearch best",     random_search, X_train, y_train, X_test, y_test))
print(pd.DataFrame(rows).set_index("model").round(3))

                          train_MAE  test_MAE  train_R2  test_R2
model                                                           
XGB baseline (no tuning)      0.785     3.503     0.919   -0.799
XGB RandomSearch best         0.554     2.962     0.951   -0.338


## 4. Grid Search

Following Random Search, we run a focused Grid Search around the best
region it found. The grid is deliberately small (3 values per parameter,
or 2 for the regularization parameters) to keep runtime manageable.

**Purpose**: verify that Random Search's best is not an artefact of random
sampling, and refine within its neighbourhood.

In [19]:
# Focused grid around the RandomSearch best
best = random_search.best_params_

# Build a small grid centered on the best values
grid_params = {
    "model__n_estimators":     [int(best["model__n_estimators"] * 0.75),
                                int(best["model__n_estimators"]),
                                int(best["model__n_estimators"] * 1.25)],
    "model__learning_rate":    [round(best["model__learning_rate"] * 0.5, 3),
                                round(best["model__learning_rate"], 3),
                                round(best["model__learning_rate"] * 1.5, 3)],
    "model__max_depth":        [max(2, best["model__max_depth"] - 1),
                                best["model__max_depth"],
                                best["model__max_depth"] + 1],
    "model__subsample":        [0.7, 0.85],
    "model__colsample_bytree": [0.7, 0.85],
    "model__reg_lambda":       [0.5, 1.0, 2.0],
}

grid_search = GridSearchCV(
    estimator=make_xgb_pipeline(),
    param_grid=grid_params,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)
print("\nBest params (Grid Search):")
for k, v in grid_search.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nBest CV MAE: {-grid_search.best_score_:.4f}")

Fitting 5 folds for each of 324 candidates, totalling 1620 fits

Best params (Grid Search):
  model__colsample_bytree: 0.7
  model__learning_rate: 0.035
  model__max_depth: 6
  model__n_estimators: 90
  model__reg_lambda: 1.0
  model__subsample: 0.85

Best CV MAE: 2.4226


### Grid Search — result

The best configuration found by Grid Search:

| Parameter | Value |
|---|---|
| `colsample_bytree` | 0.70 |
| `learning_rate` | 0.035 |
| `max_depth` | 6 |
| `n_estimators` | 90 |
| `reg_lambda` | 1.0 |
| `subsample` | 0.85 |

**Best cross-validated MAE: 2.4226** — an improvement over Random Search
(2.4916).

Grid Search settled on a *simpler* configuration than Random Search:
fewer trees, shallower depth, lower learning rate. This is consistent with
the overfitting pattern observed throughout the project: **less capacity
often generalizes better** on this dataset.

In [20]:
rows = []
rows.append(report("XGB baseline",         models.build_xgboost_baseline(), X_train, y_train, X_test, y_test))
rows.append(report("XGB RandomSearch",     random_search, X_train, y_train, X_test, y_test))
rows.append(report("XGB GridSearch",       grid_search,   X_train, y_train, X_test, y_test))
print(pd.DataFrame(rows).set_index("model").round(3))

                  train_MAE  test_MAE  train_R2  test_R2
model                                                   
XGB baseline          0.785     3.503     0.919   -0.799
XGB RandomSearch      0.554     2.962     0.951   -0.338
XGB GridSearch        1.148     2.841     0.818   -0.332


Comparing the two tuned models on the test set:

| Model | Train MAE | Test MAE | Train R² | Test R² |
|---|---|---|---|---|
| XGB baseline | 0.79 | 3.50 | 0.92 | −0.80 |
| XGB RandomSearch | 0.55 | 2.96 | 0.95 | −0.34 |
| XGB GridSearch | 1.15 | **2.84** | 0.82 | **−0.33** |

Grid Search's model has *higher* training error (1.15 vs 0.55) but *lower*
test error (2.84 vs 2.96). A less eager model generalizes slightly better.

## 5. Manual tuning

We hand-pick four extreme configurations to explore the bias-variance
boundary directly:

| Configuration | Intent |
|---|---|
| `shallow_strong_reg` | Very simple model, heavy regularization → low variance |
| `deep_weak_reg` | Deep trees, no regularization → high variance |
| `slow_learner` | 1000 trees at learning_rate=0.01 → careful gradient descent |
| `fast_learner` | 100 trees at learning_rate=0.30 → aggressive gradient descent |

The goal is to observe **how** model complexity and learning dynamics
affect the train-test gap.

In [21]:
manual_configs = {
    "shallow_strong_reg": dict(n_estimators=300, learning_rate=0.05, max_depth=2,
                                subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0),
    "deep_weak_reg":      dict(n_estimators=300, learning_rate=0.05, max_depth=8,
                                subsample=0.8, colsample_bytree=0.8, reg_lambda=0.0),
    "slow_learner":       dict(n_estimators=1000, learning_rate=0.01, max_depth=4,
                                subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0),
    "fast_learner":       dict(n_estimators=100, learning_rate=0.3, max_depth=4,
                                subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0),
}

def make_manual_pipeline(**params):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
        ("model",   XGBRegressor(random_state=config.RANDOM_STATE,
                                  n_jobs=-1, verbosity=0, **params)),
    ])

for name, cfg in manual_configs.items():
    m = make_manual_pipeline(**cfg)
    rows.append(report(f"manual: {name}", m, X_train, y_train, X_test, y_test))

print(pd.DataFrame(rows).set_index("model").round(3))

                            train_MAE  test_MAE  train_R2  test_R2
model                                                             
XGB baseline                    0.785     3.503     0.919   -0.799
XGB RandomSearch                0.554     2.962     0.951   -0.338
XGB GridSearch                  1.148     2.841     0.818   -0.332
manual: shallow_strong_reg      1.675     3.056     0.585   -0.514
manual: deep_weak_reg           0.029     3.350     1.000   -0.660
manual: slow_learner            1.038     3.331     0.856   -0.672
manual: fast_learner            0.387     3.802     0.982   -1.100


### Manual tuning — interpretation

| Config | Train MAE | Test MAE | Train R² | Test R² |
|---|---|---|---|---|
| baseline | 0.79 | 3.50 | 0.92 | −0.80 |
| RandomSearch | 0.55 | 2.96 | 0.95 | −0.34 |
| GridSearch | 1.15 | 2.84 | 0.82 | −0.33 |
| shallow_strong_reg | 1.68 | 3.06 | 0.59 | −0.51 |
| deep_weak_reg | **0.03** | 3.35 | **1.00** | −0.66 |
| slow_learner | 1.04 | 3.33 | 0.86 | −0.67 |
| fast_learner | 0.39 | 3.80 | 0.98 | −1.10 |


### Summary of the tuning tournament

| Category | Winner | Value |
|---|---|---|
| Best test MAE (all configurations) | Grid Search | 2.841 |
| Smallest train-test MAE gap | shallow_strong_reg | 1.38 |
| Worst overfitting | deep_weak_reg | train R² = 1.000, test R² = −0.66 |
| Worst test performance | fast_learner | test MAE = 3.802 |
| Best balance of simplicity and accuracy | Grid Search | 2.841 test MAE, 1.69 gap |

The Grid Search configuration wins on the primary metric (test MAE). The
manual configurations confirm the underlying pattern: the two extremes of
the bias-variance spectrum — perfectly overfit (`deep_weak_reg`) and overly
constrained — both underperform, and the best trade-off lies in between.
Key observations:

1. **`deep_weak_reg` achieves a perfect training R² of 1.000** — it has
   completely memorized the training data — yet its test R² of −0.66 is
   worse than the naive mean baseline. This is the clearest possible
   illustration of overfitting.

2. **`shallow_strong_reg` generalizes best among the manual group.**
   Its training fit is the weakest (train R² 0.59) but its test error is
   lowest among the manual configs (test MAE 3.06). Small capacity,
   strong regularization → smaller train-test gap.

3. **`slow_learner` (1000 trees at lr=0.01) beats `fast_learner`
   (100 trees at lr=0.30)** on both training and test performance.
   A low learning rate acts as a form of regularization — the model takes
   smaller, more careful steps toward the optimum.

4. **The pattern is monotonic**: as capacity increases and regularization
   decreases, training fit improves while test fit degrades.

These results directly answer the project guideline's requirement to
"understand how changing hyperparameters affects model performance and
overfitting."

## 6. Final comparison

We consolidate all tuning results into a single table and save it for
use in the evaluation notebook. The held-out test period (2020–2024)
is included here only as a **read-only comparison** — we do not select
a model based on its test performance, only report it.

The final model choice is made in `06_evaluation_and_findings.ipynb`.

In [22]:
tuning_results = pd.DataFrame(rows).set_index("model").round(4)
tuning_results.to_csv(config.PROCESSED_DIR / "tuning_results.csv")
print("Saved.")

Saved.


## Summary

Hyperparameter tuning was performed in three stages (Random Search,
Grid Search, Manual). The test set was never used for tuning decisions.

**Main findings**:

- **Tuning improved the model meaningfully.** Test MAE fell from 3.50
  (baseline) to 2.84 (GridSearch best) — a 19% reduction. This supports
  Hypothesis 3 of the project.

- **Grid Search landed on a simpler model than Random Search.** Fewer
  trees, lower learning rate, and less depth gave better test performance,
  even though training fit was weaker. Less capacity → less overfitting.

- **Manual tuning confirmed the pattern.** `deep_weak_reg` overfit
  perfectly (train R² = 1.000, test R² = −0.66). `shallow_strong_reg`
  generalized best of the manual configs. A low learning rate
  (`slow_learner`) beat a high one (`fast_learner`).

- **But tuning did not overcome the fundamental problem.** Even the
  tuned XGBoost (test MAE 2.84) still does not beat the naive mean
  baseline (2.49) or Linear Regression (2.75). This supports the overall
  project finding that next-year GDP growth is not reliably predictable
  from these indicators.

### Quantitative summary

- **Improvement from tuning**: test MAE reduced from 3.503 (baseline) to 2.841 (Grid Search best) = **18.9% reduction**.
- **Simplicity effect**: Grid Search chose a smaller model (90 trees, depth 6) than Random Search (121 trees, depth 7) and performed better on test.
- **Overfitting cost**: the unregularized config `deep_weak_reg` achieved train R² = 1.000 but test R² = −0.66 — a direct demonstration that a model can fit training data perfectly and still be useless for forecasting.
- **Ceiling of tuning**: even the best tuned model (2.841 test MAE) still loses to the naive mean baseline (2.490) and Linear Regression (2.748).

These results support the project guideline's key requirement: to understand
how hyperparameter changes affect model performance and overfitting, not
only to find the "best" parameters.

The final model is selected in `06_evaluation_and_findings.ipynb`, where
we conduct the final evaluation on the held-out test period, feature
importance, GDP-per-capita group analysis, and country-level error
diagnostics.